# 03 · Teacher caching  (step 4, GPU session, run once)
Each teacher runs once on the training images (and on validation images for the gate); outputs are
packed into one aligned uint8 cache. Save `/kaggle/working/teacher_cache` as Dataset **pedger-teachers**.
Log the GPU-hours of this notebook: they count towards the method's training cost.
BSDS teachers are trained on BSDS train+val, so their maps on these images are *in-sample*
(sharper than on unseen images). `--flip` averages with a flipped pass to soften them (optional).

In [ ]:
# --- setup: code + upstream PEdger -------------------------------------------
import os, subprocess, sys
REPO_URL = ""          # e.g. "https://github.com/<you>/pedger-mtkd.git" (or attach the code as a Kaggle Dataset)
CODE_DATASET = "/kaggle/input/pedger-mtkd-code"
W = "/kaggle/working"
REPO = f"{W}/pedger-mtkd"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run(["cp", "-r", CODE_DATASET, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py --teachers
!pip install -q -r requirements.txt

In [ ]:
# --- inputs (attach these Kaggle Datasets; names are suggestions) --------------
RAW_BSDS = "/kaggle/input/bsds500-raw/BSR/BSDS500/data"     # original Berkeley release
DATA = "/kaggle/input/pedger-data"                           # output of 01 (array stores), saved as a Dataset
CACHE = "/kaggle/input/pedger-teachers"                      # output of 03 (teacher cache), saved as a Dataset
RUNS = "/kaggle/working/runs"
import os
if not os.path.exists(DATA): DATA = "/kaggle/working/data_store"
if not os.path.exists(CACHE): CACHE = "/kaggle/working/teacher_cache"

In [ ]:
import time; T0 = time.time()
SPLIT = "trainval"           # use "train" too if you run dev-split ablations
!python teachers/export_images.py --store {DATA}/bsds --split {SPLIT} --out /kaggle/working/work/img_{SPLIT} --flip

In [ ]:
# SAUGE (SAM ViT-B frozen + side network; weights in its repo)
!pip install -q einops
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P /kaggle/working
for suffix in ["", "_flip"]:
    !python teachers/run_sauge.py --images /kaggle/working/work/img_{SPLIT}{suffix} --out /kaggle/working/work/sauge_{SPLIT}{suffix} --sam-ckpt /kaggle/working/sam_vit_b_01ec64.pth

In [ ]:
# MuGE (weights on Google Drive)
!pip install -q efficientnet_pytorch gdown
!gdown -q 15NucsEeHAFwo5O2s11pMR1BikoUtiuUX -O /kaggle/working/muge_bsds.pth
for suffix in ["", "_flip"]:
    !python teachers/run_muge.py --images /kaggle/working/work/img_{SPLIT}{suffix} --out /kaggle/working/work/muge_{SPLIT}{suffix} --ckpt /kaggle/working/muge_bsds.pth

In [ ]:
# MEMO (BSDS LoRA weights + synthetic base model on Hugging Face), via its own CLI
%cd third_party/MEMO_Edge_Detection
!pip install -q -r requirements.txt
!python download_checkpoints.py --model bsds-late --model synthetic-late
for suffix in ["", "_flip"]:
    !python edge_prediction.py --test_folder /kaggle/working/work/img_{SPLIT}{suffix} --save_folder /kaggle/working/work/memo_{SPLIT}{suffix}  --config_file configs/discrete_BSDS_finetune/binary_lora_default.yaml  --model_path pretrained_models/MEMO_BSDS_ft_late/mp_rank_00_model_states.pt  --base_model_path pretrained_models/MEMO_synthetic_late/mp_rank_00_model_states.pt --guidance_scale 1.4 --max_steps 20
%cd /kaggle/working/pedger-mtkd

In [ ]:
W = "/kaggle/working/work"
for t, sub in [("sauge", ""), ("muge", ""), ("memo", "/prediction")]:
    !python teachers/pack_cache.py --store {DATA}/bsds --split {SPLIT} --teacher {t} --pred-dir {W}/{t}_{SPLIT}{sub} --pred-dir-flip {W}/{t}_{SPLIT}_flip{sub} --out /kaggle/working/teacher_cache/bsds
for mode in ["mean", "confidence"]:
    !python teachers/ensemble.py --teacher-root /kaggle/working/teacher_cache/bsds --split {SPLIT} --teachers sauge muge memo --mode {mode}
print(f"teacher caching GPU-hours: {(time.time() - T0) / 3600:.2f}")

In [ ]:
# Alignment check (step 6 gate): red = teacher, green = label; they must trace the same contours
!python data/visualize_samples.py --store {DATA}/bsds --split {SPLIT} --n 8 --teacher-root /kaggle/working/teacher_cache/bsds --teacher ensemble_mean --augmented --out /kaggle/working/checks/aug_align.png
from IPython.display import Image; Image("/kaggle/working/checks/aug_align.png")

## NYUDv2 / MultiCue: teacher gate
Run the teachers on the dataset's training images (pack as above, with `--fit pad32` for SAUGE/MuGE)
and on its validation images (`export_images.py --eval-split val`), then:
```
!python teachers/teacher_gate.py --store {DATA}/nyud --split val --raw-root RAW_NYUD \
    --teacher sauge=.../val_sauge --teacher muge=.../val_muge --teacher memo=.../val_memo/prediction \
    --teacher ensemble=.../val_ensemble --baseline-scores runs/nyud_baseline_s0/scores_val_ns_m.json \
    --out /kaggle/working/teacher_cache/nyud/gate.json
```
Rebuild `ensemble_mean` from the kept teachers only.